# Find possible client duplicates using the Levenshtein Edit Distance

In [1]:
# Install a conda package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install python-Levenshtein


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /home/gbadmin/guestbook_env/bin/python -m pip install --upgrade pip


In [2]:
import psycopg2 as pg
import pandas.io.sql as psql
import pandas as pd
import Levenshtein
import GuestbookDbConnect as gdb
conn = gdb.guestbookDbConnect()

Welcome to Jupyter Notebook.  You are connected to the Opportunity House guestbook database!


In [3]:
def possibleduplicate(id, dupid):

  setdupidsql = "UPDATE guestbook_person \
                 SET possibleduplicateid={} \
                 WHERE idperson={}".format(id,dupid)

  cur = conn.cursor()
  cur.execute(setdupidsql)
  # get the number of updated rows
  updated_rows = cur.rowcount
  # Commit the changes to the database
  conn.commit()

possibleduplicate(17,17)

In [ ]:
# if both of the following are 0, then exact matches only will be found
maxFirstDelta = 2
maxLastDelta  = 2

person_query = "select idperson, firstname, lastname \
         from guestbook_person \
         order by idperson"

print(person_query)

data = pd.read_sql(person_query, conn)

data.head(100)
print(data)

clientlist = []

for n in range(0, data.shape[0]):
#for n in range(0, 60):  
  id = data.iloc[n,0] 
  lastname  = data.iloc[n,2]
  firstname = data.iloc[n,1]
  client = {}
  client['id'] = id 
  client['first'] = firstname
  client['last'] = lastname
  client['dups'] = []
  #print("{} {}".format(lastname, firstname))
  for m in range(0, data.shape[0]):
    if m!=n:
      dupid = data.iloc[m,0] 
      duplastname  = data.iloc[m,2]
      dupfirstname = data.iloc[m,1]
      last  = Levenshtein.distance(lastname, duplastname)
      first = Levenshtein.distance(firstname, dupfirstname)
      if (last<=maxLastDelta) and (first<=maxFirstDelta):
        #cid is a possible duplication
        dup = {}
        dup['id'] = dupid 
        dup['first'] = dupfirstname
        dup['last'] = duplastname
        #print("  {} {}".format(clastname, cfirstname))
        client['dups'].append(dup)
        #write the cid into the possibleduplicateid field
        possibleduplicate(id, dupid)
        #only recording the first duplicate, break out of loop
        break
        
  if (len(client['dups'])>0):
    clientlist.append(client)  
#print(clientlist)


select idperson, firstname, lastname          from guestbook_person          order by idperson
      idperson firstname  lastname
0           17   CHARLIE     PRICE
1           18    JORDAN   SIMMONS
2           19     LANCE   WHITLEY
3           20      JOHN     WAYNE
4           21    THOMAS     LUKER
...        ...       ...       ...
3465      3557    CAMDEN    WILHIT
3466      3558    ASHLEY    ABBOTT
3467      3559   KENNETH     SMITH
3468      3560      ERIC       KEE
3469      3561  JONATHAN  HILTONEN

[3470 rows x 3 columns]


/tmp/ipykernel_4152213/3280017882.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(person_query, conn)


In [ ]:
for client in clientlist:
    print("\n{}..{}..{}".format(client['id'], client['first'], client['last']))
    for duplicate in client['dups']:
        print("   {}..{}..{}".format(duplicate['id'], duplicate['first'], duplicate['last']))

In [ ]:
person_query = "select idperson, firstname, lastname \
  from guestbook_person \
  where (firstname, lastname) in ( \
     select firstname, lastname \
       from guestbook_person \
       group by firstname, lastname \
       having count(*) > 1 \
     ) \
  order by lastname, firstname" 

print(person_query)

data = pd.read_sql(person_query, conn)

#data.head(100)

for n in range(0, data.shape[0]):  
  id = data.iloc[n,0] 
  lastname  = data.iloc[n,2]
  firstname = data.iloc[n,1]  
  print("{} {}\n".format(id,lastname,firstname))